In [1]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from collections import deque
import winsound, wave, struct, math, time, csv, os, urllib.request

IMG_SIZE = 96
SEQ_LEN = 5
MODEL_WEIGHTS_PATH = "C:/Users/GOPICHAND/Desktop/InternSpark/TASK-2/drowsiness_model_final_v2.weights.h5"
TARGET_MEAN = 128.0
PREDICTION_THRESHOLD = 0.5
FRAME_SKIP = 5

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# --- DNN face detector ---
os.makedirs("dnn_face_model", exist_ok=True)
files = {
    "dnn_face_model/deploy.prototxt": "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
    "dnn_face_model/res10_300x300_ssd_iter_140000.caffemodel": "https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"
}
for path, url in files.items():
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
dnn_face_net = cv2.dnn.readNetFromCaffe("dnn_face_model/deploy.prototxt",
                                        "dnn_face_model/res10_300x300_ssd_iter_140000.caffemodel")

def detect_face_dnn(frame_bgr, conf_threshold=0.5):
    h, w = frame_bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(frame_bgr, 1.0, (300, 300), (104.0, 177.0, 123.0))
    dnn_face_net.setInput(blob)
    detections = dnn_face_net.forward()
    best_box, best_conf = None, 0
    for i in range(detections.shape[2]):
        conf = detections[0, 0, i, 2]
        if conf > conf_threshold and conf > best_conf:
            box = detections[0, 0, i, 3:7] * [w, h, w, h]
            best_box, best_conf = box.astype(int), conf
    return best_box

def preprocess_frame(frame_bgr, padding=0.25):
    gray_full = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray_full.shape
    box = detect_face_dnn(frame_bgr)
    if box is None:
        side = min(h, w); cy, cx = h // 2, w // 2
        y1, y2 = max(0, cy - side // 2), min(h, cy + side // 2)
        x1, x2 = max(0, cx - side // 2), min(w, cx + side // 2)
        cropped = gray_full[y1:y2, x1:x2]; face_box = None
    else:
        x1, y1, x2, y2 = box
        fw, fh = x2 - x1, y2 - y1
        pad_w, pad_h = int(fw * padding), int(fh * padding)
        x1, y1 = max(0, x1 - pad_w), max(0, y1 - pad_h)
        x2, y2 = min(w, x2 + pad_w), min(h, y2 + pad_h)
        cropped = gray_full[y1:y2, x1:x2]; face_box = (x1, y1, x2, y2)
    equalized = clahe.apply(cropped)
    current_mean = equalized.mean()
    if current_mean > 0:
        equalized = np.clip(equalized.astype(np.float32) + (TARGET_MEAN - current_mean), 0, 255).astype(np.uint8)
    resized = cv2.resize(equalized, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return resized.astype(np.float32) / 255.0, face_box

# --- Model architecture + load trained weights ---
class AttentionLayer(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        self.W = self.add_weight(name="att_W", shape=(input_shape[-1], self.units), initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="att_b", shape=(self.units,), initializer="zeros", trainable=True)
        self.u = self.add_weight(name="att_u", shape=(self.units, 1), initializer="glorot_uniform", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        score = tf.nn.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        score = tf.tensordot(score, self.u, axes=1)
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context, weights

def build_model_transfer():
    inputs = layers.Input(shape=(SEQ_LEN, IMG_SIZE, IMG_SIZE, 1))
    rgb = layers.TimeDistributed(layers.Lambda(lambda t: tf.repeat(t, 3, axis=-1)))(inputs)
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=None, pooling='avg')
    x = layers.TimeDistributed(base)(rgb)
    x = layers.TimeDistributed(layers.Dense(128, activation='relu'))(x)
    x = layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)(x)
    x = layers.Dropout(0.4)(x)
    context, attn_weights = AttentionLayer(units=64)(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(context)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs=inputs, outputs=outputs)

model = build_model_transfer()
model.load_weights(MODEL_WEIGHTS_PATH)
print("Model loaded.")

# --- Alarm sound ---
def generate_alarm_wav(path="alarm.wav", freq=1000, duration=0.5, sample_rate=44100):
    n_samples = int(duration * sample_rate)
    with wave.open(path, 'w') as wav_file:
        wav_file.setnchannels(1); wav_file.setsampwidth(2); wav_file.setframerate(sample_rate)
        for i in range(n_samples):
            value = int(32767 * math.sin(2 * math.pi * freq * i / sample_rate))
            wav_file.writeframesraw(struct.pack('<h', value))

if not os.path.exists("alarm.wav"):
    generate_alarm_wav("alarm.wav")

# --- MediaPipe FaceLandmarker (Tasks API — this mediapipe build has no mp.solutions) ---
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

face_landmarker_path = "face_landmarker.task"
if not os.path.exists(face_landmarker_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        face_landmarker_path
    )
    print("Downloaded face_landmarker.task")

base_options = mp_python.BaseOptions(model_asset_path=face_landmarker_path)
landmarker_options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.VIDEO,
    num_faces=1
)
landmarker = mp_vision.FaceLandmarker.create_from_options(landmarker_options)

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH_TOP, MOUTH_BOTTOM, MOUTH_LEFT, MOUTH_RIGHT = 13, 14, 61, 291

EAR_THRESHOLD = 0.21
MAR_THRESHOLD = 0.55
EAR_SUSTAIN_FRAMES = 15
MAR_SUSTAIN_FRAMES = 10

def euclidean(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def compute_ear(landmarks, eye_indices, w, h):
    pts = [(landmarks[i].x * w, landmarks[i].y * h) for i in eye_indices]
    vertical1 = euclidean(pts[1], pts[5])
    vertical2 = euclidean(pts[2], pts[4])
    horizontal = euclidean(pts[0], pts[3])
    return (vertical1 + vertical2) / (2.0 * horizontal) if horizontal > 0 else 0

def compute_mar(landmarks, w, h):
    top = (landmarks[MOUTH_TOP].x * w, landmarks[MOUTH_TOP].y * h)
    bottom = (landmarks[MOUTH_BOTTOM].x * w, landmarks[MOUTH_BOTTOM].y * h)
    left = (landmarks[MOUTH_LEFT].x * w, landmarks[MOUTH_LEFT].y * h)
    right = (landmarks[MOUTH_RIGHT].x * w, landmarks[MOUTH_RIGHT].y * h)
    vertical = euclidean(top, bottom)
    horizontal = euclidean(left, right)
    return vertical / horizontal if horizontal > 0 else 0

_frame_timestamp_ms = 0

def compute_ear_mar(frame_bgr):
    global _frame_timestamp_ms
    h, w = frame_bgr.shape[:2]
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    _frame_timestamp_ms += 33
    result = landmarker.detect_for_video(mp_image, _frame_timestamp_ms)
    if not result.face_landmarks:
        return None, None
    landmarks = result.face_landmarks[0]
    left_ear = compute_ear(landmarks, LEFT_EYE, w, h)
    right_ear = compute_ear(landmarks, RIGHT_EYE, w, h)
    ear = (left_ear + right_ear) / 2.0
    mar = compute_mar(landmarks, w, h)
    return ear, mar

print("Setup complete.")


Model loaded.
Setup complete.


In [7]:
# ============================================================
# FINAL SIMPLE WEBCAM DISPLAY
# Detection logic remains unchanged
# ============================================================

def run_session(
    log_ground_truth=False,
    use_alarm=True,
    log_path="test_session_log.csv"
):

    # --------------------------------------------------------
    # Temporal CNN buffer
    # --------------------------------------------------------
    frame_buffer = deque(maxlen=SEQ_LEN)

    # --------------------------------------------------------
    # Open webcam
    # --------------------------------------------------------
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        raise RuntimeError("Could not open webcam.")

    # --------------------------------------------------------
    # Counters / states
    # --------------------------------------------------------
    frame_counter = 0

    consecutive_drowsy_cnn = 0
    consecutive_eyes_closed = 0
    consecutive_yawn = 0
    consecutive_clearly_alert = 0

    alarm_playing = False

    current_ground_truth = "notdrowsy"

    current_state = "AWAKE"

    log_rows = []

    # --------------------------------------------------------
    # Clear-alert recovery settings
    # --------------------------------------------------------
    CLEARLY_ALERT_EAR = EAR_THRESHOLD + 0.05
    CLEARLY_ALERT_MAR = MAR_THRESHOLD - 0.10
    CLEARLY_ALERT_FRAMES = 5

    # --------------------------------------------------------
    # Instructions
    # --------------------------------------------------------
    if log_ground_truth:

        print(
            "Press 'd' = ground truth DROWSY"
            "\nPress 'a' = ground truth ALERT"
            "\nPress 'q' = quit & save."
        )

    else:

        print("Press 'q' to quit.")

    # ========================================================
    # MAIN LOOP
    # ========================================================

    while True:

        ret, frame = cap.read()

        if not ret:
            break


        # ====================================================
        # PREPROCESS FRAME
        # ====================================================

        processed, face_box = preprocess_frame(frame)

        frame_counter += 1

        if frame_counter % FRAME_SKIP == 0:
            frame_buffer.append(processed)


        # ====================================================
        # EAR / MAR HEURISTIC BRANCH
        # ====================================================

        ear, mar = compute_ear_mar(frame)

        eyes_closed_now = False
        yawn_now = False
        clearly_alert_now = False

        if ear is not None:

            eyes_closed_now = (
                ear < EAR_THRESHOLD
            )

            yawn_now = (
                mar > MAR_THRESHOLD
            )

            clearly_alert_now = (
                ear > CLEARLY_ALERT_EAR
                and
                mar < CLEARLY_ALERT_MAR
            )


        # ----------------------------------------------------
        # Maintain consecutive counters
        # ----------------------------------------------------

        consecutive_eyes_closed = (
            consecutive_eyes_closed + 1
            if eyes_closed_now
            else 0
        )

        consecutive_yawn = (
            consecutive_yawn + 1
            if yawn_now
            else 0
        )

        consecutive_clearly_alert = (
            consecutive_clearly_alert + 1
            if clearly_alert_now
            else 0
        )


        # ----------------------------------------------------
        # Heuristic drowsiness
        # ----------------------------------------------------

        heuristic_drowsy = (
            consecutive_eyes_closed
            >= EAR_SUSTAIN_FRAMES
        ) or (
            consecutive_yawn
            >= MAR_SUSTAIN_FRAMES
        )


        # ====================================================
        # FAST RECOVERY
        # ====================================================

        if (
            consecutive_clearly_alert
            >= CLEARLY_ALERT_FRAMES
        ):

            frame_buffer.clear()

            consecutive_drowsy_cnn = 0


        # ====================================================
        # CNN BRANCH
        # ====================================================

        cnn_drowsy = False

        if len(frame_buffer) == SEQ_LEN:

            X = np.stack(
                frame_buffer,
                axis=0
            )[np.newaxis, ..., np.newaxis]


            score = float(
                model.predict(
                    X,
                    verbose=0
                )[0][0]
            )


            pred_label = (
                "drowsy"
                if score >= PREDICTION_THRESHOLD
                else "notdrowsy"
            )


            # ------------------------------------------------
            # CNN persistence
            # ------------------------------------------------

            if pred_label == "drowsy":

                consecutive_drowsy_cnn += 1

            else:

                consecutive_drowsy_cnn = 0


            cnn_drowsy = (
                consecutive_drowsy_cnn >= 3
            )


            # ------------------------------------------------
            # Logging
            # ------------------------------------------------

            if log_ground_truth:

                log_rows.append({
                    "timestamp": time.time(),
                    "predicted": pred_label,
                    "score": score,
                    "ground_truth": current_ground_truth,
                    "face_detected": face_box is not None
                })


        # ====================================================
        # FINAL DECISION
        # ====================================================

        overall_drowsy = (
            cnn_drowsy
            or
            heuristic_drowsy
        )


        # ====================================================
        # STATE
        # ====================================================

        if overall_drowsy:

            current_state = "DROWSY"

        else:

            current_state = "AWAKE"


        # ====================================================
        # ALARM
        # ====================================================

        if use_alarm:

            if (
                overall_drowsy
                and
                not alarm_playing
            ):

                winsound.PlaySound(
                    "alarm.wav",
                    winsound.SND_FILENAME
                    |
                    winsound.SND_ASYNC
                    |
                    winsound.SND_LOOP
                )

                alarm_playing = True


            if (
                not overall_drowsy
                and
                alarm_playing
            ):

                winsound.PlaySound(
                    None,
                    winsound.SND_PURGE
                )

                alarm_playing = False


        # ====================================================
        # MINIMAL DISPLAY
        # ====================================================

        if overall_drowsy:

            status_color = (
                0,
                0,
                255
            )

            status_text = "DROWSY"

        else:

            status_color = (
                0,
                255,
                0
            )

            status_text = "AWAKE"


        # ----------------------------------------------------
        # Face rectangle
        # ----------------------------------------------------

        if face_box is not None:

            x1, y1, x2, y2 = face_box

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                status_color,
                2
            )


        # ----------------------------------------------------
        # Main status
        # ----------------------------------------------------

        cv2.putText(
            frame,
            status_text,
            (20, 55),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.3,
            status_color,
            4
        )


        # ----------------------------------------------------
        # Alarm message
        # ----------------------------------------------------

        if alarm_playing:

            cv2.putText(
                frame,
                "WAKE UP!",
                (20, 105),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (0, 0, 255),
                3
            )


        # ====================================================
        # SHOW WINDOW
        # ====================================================

        cv2.imshow(
            "Driver Drowsiness Detection",
            frame
        )


        # ====================================================
        # KEYBOARD INPUT
        # ====================================================

        key = (
            cv2.waitKey(1)
            & 0xFF
        )


        if key == ord("q"):

            break


        # ----------------------------------------------------
        # Ground truth logging controls
        # ----------------------------------------------------

        elif (
            log_ground_truth
            and key == ord("d")
        ):

            current_ground_truth = "drowsy"

            print(
                "Ground truth -> DROWSY"
            )


        elif (
            log_ground_truth
            and key == ord("a")
        ):

            current_ground_truth = "notdrowsy"

            print(
                "Ground truth -> ALERT"
            )


    # ========================================================
    # CLEANUP
    # ========================================================

    winsound.PlaySound(
        None,
        winsound.SND_PURGE
    )

    cap.release()

    cv2.destroyAllWindows()


    # ========================================================
    # SAVE LOG
    # ========================================================

    if log_ground_truth:

        with open(
            log_path,
            "w",
            newline=""
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "timestamp",
                    "predicted",
                    "score",
                    "ground_truth",
                    "face_detected"
                ]
            )

            writer.writeheader()

            writer.writerows(
                log_rows
            )


        print(
            f"Saved {len(log_rows)} "
            f"predictions to {log_path}"
        )

Press 'q' to quit.


In [8]:
run_session(log_ground_truth=False, use_alarm=True)

Press 'q' to quit.
